# Track a new clip with SAM 2

Produces the cached tracking arrays for a **new** tennis clip, so the analysis in
`player_ball_tracking.ipynb` can be run against something other than the sample point.

**Run this on Colab with a GPU runtime** (`Runtime -> Change runtime type -> T4 GPU`).
SAM 2 video inference is not practical on CPU: the sample clip took ~15 minutes on a T4.

## What you get

A `tracking_<clip>.zip` containing:

| file | what it is |
|---|---|
| `rectangles_p1.npy`, `rectangles_p2.npy`, `rectangles_ball.npy` | `(T, 4)` boxes `[x1, y1, x2, y2]` per frame |
| `clip_meta.json` | fps, the court corners, and **`frame_offset`** |

`frame_offset` is the one that matters. Array index `i` is **not** video frame `i` — it is
video frame `i + frame_offset`, because frames are extracted from `START_IDX` and SAM 2
propagates forward from the frame you annotate. On the sample clip that offset is 147, it
was not recorded anywhere, and rediscovering it took a background-matching search. This
notebook writes it down at the point it is created, and section 8 checks it visually.

## Why bother

The current detector is validated on one clip with no volleys. `bounce.py` assumes bounces
and racket contacts strictly alternate, so a volley makes it invent a bounce. **Pick a clip
containing at least one volley** if you want that limitation tested rather than restated.


## 1. Check the GPU

If this errors or shows no GPU, change the runtime type before going further.

In [ ]:
!nvidia-smi

## 2. Install SAM 2

Only the `large` checkpoint is downloaded (~857 MB). The original notebook pulled all
four sizes, ~1.5 GB, and then used only this one.

In [ ]:
import os

HOME = os.getcwd()
print("HOME:", HOME)

In [ ]:
!git clone -q https://github.com/facebookresearch/segment-anything-2.git
%cd {HOME}/segment-anything-2
!pip install -e . -q
!python setup.py build_ext --inplace
%cd {HOME}

In [ ]:
!pip install -q supervision[assets] jupyter_bbox_widget

In [ ]:
!mkdir -p {HOME}/checkpoints
!wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt -P {HOME}/checkpoints

## 3. Imports and model

In [ ]:
import base64
import json
from pathlib import Path

import cv2
import numpy as np
import torch
import supervision as sv
from sam2.build_sam import build_sam2_video_predictor

from google.colab import output
output.enable_custom_widget_manager()
from jupyter_bbox_widget import BBoxWidget

In [ ]:
torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT = f"{HOME}/checkpoints/sam2_hiera_large.pt"
CONFIG = "sam2_hiera_l.yaml"

sam2_model = build_sam2_video_predictor(CONFIG, CHECKPOINT)
print("model loaded on", DEVICE)

## 4. The clip

Upload a clip of **one point**, from a standard broadcast camera behind the baseline.
Longer clips cost proportionally more tracking time and gain nothing.

In [ ]:
from google.colab import files

uploaded = files.upload()
SOURCE_VIDEO = f"/content/{next(iter(uploaded))}"
print(SOURCE_VIDEO)

In [ ]:
video_info_src = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
print(video_info_src)
print(f"duration: {video_info_src.total_frames / video_info_src.fps:.1f} s")

### Choose the segment

`START_IDX` / `END_IDX` bound the frames extracted for tracking, and `SCALE_FACTOR`
shrinks them to save VRAM.

Trim tightly to the point itself. Every extracted frame is held in GPU memory by the
inference state, so a loose range is the usual cause of an out-of-memory crash.

In [ ]:
SCALE_FACTOR = 1.0
START_IDX = 0
END_IDX = 700

print(f"{END_IDX - START_IDX} frames "
      f"({(END_IDX - START_IDX) / video_info_src.fps:.1f} s) will be extracted")

In [ ]:
SOURCE_FRAMES = Path(HOME) / Path(SOURCE_VIDEO).stem
SOURCE_FRAMES.mkdir(parents=True, exist_ok=True)

frames_generator = sv.get_video_frames_generator(SOURCE_VIDEO, start=START_IDX, end=END_IDX)
images_sink = sv.ImageSink(
    target_dir_path=SOURCE_FRAMES.as_posix(),
    overwrite=True,
    image_name_pattern="{:05d}.jpeg",
)

with images_sink:
    for frame in frames_generator:
        images_sink.save_image(sv.scale_image(frame, SCALE_FACTOR))

SOURCE_FRAME_PATHS = sorted(
    sv.list_files_with_extensions(SOURCE_FRAMES.as_posix(), extensions=["jpeg"])
)
print(f"extracted {len(SOURCE_FRAME_PATHS)} frames")

## 5. Annotate the reference frame

SAM 2 needs one box per object on a single frame, and tracks forward from there.

**Annotate as early as you can.** Propagation only runs forward, so anything before
`FRAME_IDX` is never tracked. `FRAME_IDX = 0` gives full coverage and an offset equal to
`START_IDX`; the sample clip used 146 and lost the first 146 frames of its segment.
Pick a later frame only if the players or ball are hard to see at the start.

In [ ]:
inference_state = sam2_model.init_state(video_path=SOURCE_FRAMES.as_posix())
sam2_model.reset_state(inference_state)

In [ ]:
OBJECTS = ["player-1", "player-2", "ball"]


def encode_image(filepath):
    with open(filepath, "rb") as f:
        encoded = str(base64.b64encode(f.read()), "utf-8")
    return "data:image/jpg;base64," + encoded

Draw **exactly one** box per label — `player-1` (near player), `player-2` (far player),
`ball`. Keep the boxes tight; a loose ball box shifts its centre and every bounce with it.

Order matters and must stay consistent: `player-1` becomes `rectangles_p1.npy`.

In [ ]:
FRAME_IDX = 0
FRAME_PATH = Path(SOURCE_FRAMES) / f"{FRAME_IDX:05d}.jpeg"

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image(FRAME_PATH)
widget

In [ ]:
labels_drawn = {box["label"] for box in widget.bboxes}
missing = [o for o in OBJECTS if o not in labels_drawn]
assert not missing, f"no box drawn for: {missing}"

for object_id, label in enumerate(OBJECTS, start=1):
    for box in [b for b in widget.bboxes if b["label"] == label]:
        input_box = np.array(
            [box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]],
            dtype=np.float32,
        )
        _, object_ids, mask_logits = sam2_model.add_new_points_or_box(
            inference_state=inference_state,
            frame_idx=FRAME_IDX,
            obj_id=object_id,
            box=input_box,
        )
print(f"prompted {len(OBJECTS)} objects on frame {FRAME_IDX}")

## 6. Track

The slow part — roughly 1.5 s per frame on a T4. It also writes an annotated video so
you can see where tracking drifted.

In [ ]:
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
video_info.width = int(video_info.width * SCALE_FACTOR)
video_info.height = int(video_info.height * SCALE_FACTOR)

COLORS = ["#FF1493", "#00BFFF", "#FF6347"]
mask_annotator = sv.MaskAnnotator(
    color=sv.ColorPalette.from_hex(COLORS), color_lookup=sv.ColorLookup.CLASS
)
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(COLORS), color_lookup=sv.ColorLookup.CLASS
)

list_xyxy = []
TRACKED_VIDEO = f"{HOME}/tracked_preview.mp4"

with sv.VideoSink(target_path=TRACKED_VIDEO, video_info=video_info) as sink:
    for frame_idx, object_ids, mask_logits in sam2_model.propagate_in_video(inference_state):
        frame = cv2.imread(SOURCE_FRAME_PATHS[frame_idx])
        masks = np.squeeze((mask_logits > 0.0).cpu().numpy()).astype(bool)
        detections = sv.Detections(
            xyxy=sv.mask_to_xyxy(masks=masks), mask=masks, class_id=np.array(object_ids)
        )
        list_xyxy.append(detections.xyxy)

        annotated = mask_annotator.annotate(scene=frame.copy(), detections=detections)
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        sink.write_frame(annotated)

print(f"tracked {len(list_xyxy)} frames -> {TRACKED_VIDEO}")

In [ ]:
data_array = np.array(list_xyxy)
assert data_array.shape[1] == len(OBJECTS), (
    f"expected {len(OBJECTS)} objects per frame, got {data_array.shape[1]}. "
    "SAM 2 usually lost an object - check the preview video."
)

rectangles_p1 = data_array[:, 0, :]
rectangles_p2 = data_array[:, 1, :]
rectangles_ball = data_array[:, 2, :]

for name, arr in [("p1", rectangles_p1), ("p2", rectangles_p2), ("ball", rectangles_ball)]:
    lost = int((arr == 0).all(axis=1).sum())
    print(f"{name:>4}: shape {arr.shape}, {lost} frames with no detection "
          f"({100 * lost / len(arr):.1f}%)")

## 7. Court corners

The homography needs the four corners of the **doubles** court in pixels. They are
per-clip: a different camera angle means different corners.

Hover the plot to read coordinates, then type them in below, clockwise from the far-left
corner. Use a frame where all four corners are unobstructed.

In [ ]:
import plotly.express as px

CORNER_FRAME = FRAME_IDX

cap = cv2.VideoCapture(SOURCE_VIDEO)
cap.set(cv2.CAP_PROP_POS_FRAMES, START_IDX + CORNER_FRAME)
ok, frame = cap.read()
cap.release()
assert ok, "could not read that frame"

fig = px.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
fig.update_layout(title="Hover to read (x, y); clockwise from the FAR-LEFT corner",
                  dragmode="zoom", hovermode="closest", height=700)
fig.show()

In [ ]:
# Clockwise from the far-left corner: far-left, far-right, near-right, near-left.
IMAGE_CORNERS = np.float32([
    [592, 362],    # far-left
    [1327, 367],   # far-right
    [1571, 897],   # near-right
    [339, 890],    # near-left
])

COURT_LENGTH, COURT_WIDTH = 23.77, 10.97
COURT_CORNERS = np.float32([[0, 0], [0, COURT_WIDTH], [COURT_LENGTH, COURT_WIDTH], [COURT_LENGTH, 0]])
H, _ = cv2.findHomography(IMAGE_CORNERS, COURT_CORNERS)

overlay = frame.copy()
cv2.polylines(overlay, [IMAGE_CORNERS.astype(np.int32).reshape(-1, 1, 2)], True, (0, 0, 255), 3)
sv.plot_image(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB), size=(14, 8))

# A player's feet should land on the court, roughly 0-5 m behind their baseline.
feet = np.float32([[(rectangles_p1[0][0] + rectangles_p1[0][2]) / 2, rectangles_p1[0][3]]])
print("player-1 feet at frame 0 ->", cv2.perspectiveTransform(feet.reshape(-1, 1, 2), H).reshape(2))
print(f"court is {COURT_LENGTH} x {COURT_WIDTH} m")

## 8. Check the frame offset

Array index `i` corresponds to video frame `i + frame_offset`. This draws the tracked
boxes on the **original** video at that mapping — if the offset is right the boxes sit on
the players. If they sit on empty court, something above changed and the offset is wrong.

Getting this wrong is not obvious later: the analysis numbers all still come out, they
just describe the wrong frames.

In [ ]:
import matplotlib.pyplot as plt

FRAME_OFFSET = START_IDX + FRAME_IDX
print(f"frame_offset = START_IDX ({START_IDX}) + FRAME_IDX ({FRAME_IDX}) = {FRAME_OFFSET}")

cap = cv2.VideoCapture(SOURCE_VIDEO)
picks = np.linspace(0, len(rectangles_p1) - 1, 4).astype(int)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))

for ax, i in zip(axes.ravel(), picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i) + FRAME_OFFSET)
    ok, img = cap.read()
    if not ok:
        continue
    for arr, colour in [(rectangles_p1, (0, 255, 0)), (rectangles_p2, (255, 128, 0)),
                        (rectangles_ball, (0, 0, 255))]:
        x1, y1, x2, y2 = arr[i].astype(int)
        if (x1, y1, x2, y2) != (0, 0, 0, 0):
            cv2.rectangle(img, (x1, y1), (x2, y2), colour, 3)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"array {i} -> video frame {i + FRAME_OFFSET}")
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

## 9. Save and download

In [ ]:
OUT = Path(HOME) / f"tracking_{Path(SOURCE_VIDEO).stem}"
OUT.mkdir(exist_ok=True)

np.save(OUT / "rectangles_p1.npy", rectangles_p1)
np.save(OUT / "rectangles_p2.npy", rectangles_p2)
np.save(OUT / "rectangles_ball.npy", rectangles_ball)

meta = {
    "clip": Path(SOURCE_VIDEO).name,
    "fps": float(video_info_src.fps),
    "width": int(video_info_src.width),
    "height": int(video_info_src.height),
    "total_frames": int(video_info_src.total_frames),
    "start_idx": int(START_IDX),
    "end_idx": int(END_IDX),
    "scale_factor": float(SCALE_FACTOR),
    "annotation_frame": int(FRAME_IDX),
    "frame_offset": int(FRAME_OFFSET),
    "n_frames_tracked": int(len(rectangles_p1)),
    "objects": OBJECTS,
    "court_corners_px": IMAGE_CORNERS.tolist(),
    "court_dims_m": [COURT_LENGTH, COURT_WIDTH],
    "note": "array index i corresponds to video frame i + frame_offset",
}
(OUT / "clip_meta.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

In [ ]:
!zip -qr {OUT}.zip {OUT}
from google.colab import files
files.download(f"{OUT}.zip")

## Next, on your own machine

1. Unzip beside the repo, keeping `clip_meta.json` with the arrays.
2. `python run_analysis.py --clip tracking_<name>` — it reads the corners and offset from
   the metadata instead of the sample clip's hardcoded values.
3. Label the real bounces with `_annotate/annotate.html`. Set `offset` in `data.js` to
   this clip's `frame_offset`.
4. Compare. **The interesting case is a rally containing a volley**: `bounce.py` will
   emit a bounce for the volleyed shot that never happened, because it assumes bounces and
   contacts alternate. A second labelled clip is what turns that from a documented guess
   into a measured failure rate — and gives you the evidence to fix it.